# 02 · Flare Listesinin Homojenleştirilmesi — Bölüm B
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.1 (05.11.2026) — `EVENT_PEAK_SATURATED` durumundaki doymuş patlamalar eklendi

**Bölüm A'nın sonuçları:**
- HEK'te farklı kaynaklardan çift kayıt sorunu **yok**: kayıtların %99,8'i tek kaynaktan (SWPC) geliyor, yalnızca 3 olası tekrar bulundu. Yani Döngü 25'teki fazlalığın sebebi çift sayım değil.
- Ama aynı kaynakta **hatalı kayıt** var: 2024-05-08 21:40'taki patlama hem X1.0 hem X9.0 olarak, ikincisi aktif bölgesiz kaydedilmiş. Gerçeği X1.0.
- NCEI'den 12 uydunun (GOES 8–19) bilimsel kalite flare özeti indi. Dosyalar satır bazlı: her patlama için `EVENT_START`, `EVENT_PEAK`, `EVENT_END` satırları ve ortak bir `flare_id`.

**Bu defter:**
1. 12 dosyayı okuyup her uydu için olay tablosu kurar.
2. Aynı patlamayı gören uyduları birleştirip tek bir homojen liste oluşturur.
3. NCEI ile HEK akılarını eşleştirip **ölçek farkını yıl yıl ölçer**. Beklenti: GOES-16 öncesi oran ≈ 1/0.7 ≈ 1,43, sonrası ≈ 1.
4. Düzeltme öncesi ve sonrası döngü karşılaştırmasını çizer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import re, json
import xarray as xr
import matplotlib.pyplot as plt, matplotlib.dates as mdates

KOK = Path('/content/drive/MyDrive/gunes_tezi')
HAM, ISLENMIS, SEKIL = KOK/'data'/'raw', KOK/'data'/'processed', KOK/'figures'
print('Hazır')

In [ ]:
# Saf fonksiyonlar (Drive'a bağlı değil, test edilebilir)
import numpy as np, pandas as pd

TABAN = {'A':1e-8, 'B':1e-7, 'C':1e-6, 'M':1e-5, 'X':1e-4}

def sinif_harf(aki):
    # Pik akıdan sınıf harfi: <1e-7 A, <1e-6 B, <1e-5 C, <1e-4 M, aksi X
    return pd.cut(np.log10(np.asarray(aki, dtype=float)), [-np.inf, -7, -6, -5, -4, np.inf],
                  labels=list('ABCMX'), right=False).astype(str)

def sinif_metin(aki):
    h = sinif_harf(aki)
    return [f'{x}{a / TABAN[x]:.1f}' if x in TABAN else None for x, a in zip(h, np.asarray(aki, float))]

def olaylara_cevir(df):
    # Satır bazlı NCEI kaydını (START/PEAK/END) olay bazlı tabloya çevirir. Anahtar: uydu + flare_id
    # GOES 8/10/12'de dedektörün doyduğu büyük patlamalar EVENT_PEAK_SATURATED olarak işaretli.
    # Bunlar da pik sayılır; akıları alt sınırdır (gerçek değer daha büyük).
    df = df.dropna(subset=['flare_id']).copy()
    key = ['uydu', 'flare_id']
    df['doymus'] = df.status == 'EVENT_PEAK_SATURATED'
    pk = (df[df.status.isin(['EVENT_PEAK', 'EVENT_PEAK_SATURATED'])].sort_values('xrsb_flux')
            .groupby(key).agg(pik=('time', 'last'), pik_aki=('xrsb_flux', 'last'),
                              arka_plan=('background_flux', 'last'), sinif_ncei=('flare_class', 'last'),
                              pik_sayisi=('time', 'size'), doymus=('doymus', 'any')))
    bas = df[df.status == 'EVENT_START'].groupby(key).time.min().rename('baslangic')
    son = df[df.status == 'EVENT_END'].groupby(key).agg(bitis=('time', 'max'),
                                                        entegre_aki=('integrated_flux', 'max'))
    o = pk.join(bas).join(son).reset_index()
    o['sinif'] = sinif_harf(o.pik_aki)
    return o

def uydulari_birlestir(olay, oncelik, pencere_dk=10, oran_siniri=3.0):
    # Aynı patlamayı gören farklı uyduların kayıtlarını tek olayda toplar.
    # Kural: pik zamanları pencere_dk içinde, uydular farklı, akı oranı 1/oran_siniri..oran_siniri.
    # Her grupta öncelik sırası en yüksek uydunun kaydı temsilci olur.
    o = olay.sort_values('pik').reset_index(drop=True)
    pik = o.pik.values.astype('datetime64[s]').astype(np.int64)
    aki = o.pik_aki.values; uy = o.uydu.values
    grup = np.empty(len(o), dtype=np.int64)
    acik = []   # [grup_no, pik_s, aki, set(uydular)]
    sayac = 0; pen = pencere_dk * 60
    for i in range(len(o)):
        acik = [g for g in acik if pik[i] - g[1] <= pen]
        hedef = None
        for g in acik:
            if uy[i] not in g[3] and 1/oran_siniri <= aki[i]/g[2] <= oran_siniri:
                hedef = g; break
        if hedef is None:
            hedef = [sayac, pik[i], aki[i], set()]; acik.append(hedef); sayac += 1
        hedef[3].add(uy[i]); grup[i] = hedef[0]
    o['grup'] = grup
    o['oncelik'] = o.uydu.map(oncelik).fillna(99)
    o['n_uydu'] = o.groupby('grup').uydu.transform('nunique')
    if 'doymus' in o:
        o['doymus'] = o.groupby('grup').doymus.transform('any')
    temsil = o.sort_values(['grup', 'oncelik']).drop_duplicates('grup').drop(columns='oncelik')
    return temsil.reset_index(drop=True), o

def hek_ile_eslestir(ncei, hek, pencere_dk=10):
    a_ = ncei.sort_values('pik')[['pik', 'pik_aki', 'sinif', 'uydu']]
    b_ = hek.sort_values('event_peaktime')[['event_peaktime', 'pik_aki', 'fl_goescls']].rename(
        columns={'event_peaktime': 'pik_hek', 'pik_aki': 'aki_hek'})
    m = pd.merge_asof(a_, b_, left_on='pik', right_on='pik_hek',
                      tolerance=pd.Timedelta(minutes=pencere_dk), direction='nearest')
    m['oran'] = m.pik_aki / m.aki_hek
    return m

## 1 · Dosyaları oku
Her dosyada aynı değişkenlerin olup olmadığı kontrol edilir. GOES 8–12 dosyaları (sürüm 1-0-0) Bölüm A'da incelenmedi; farklı bir yapı çıkarsa hücre bunu yazar ve o dosyayı atlar.

In [ ]:
GEREKLI = ['xrsb_flux', 'status', 'background_flux', 'flare_class', 'integrated_flux', 'flare_id']
parcalar, atlanan = [], []
for p in sorted((HAM/'ncei'/'flsum').glob('*.nc')):
    uydu = int(re.search(r'_g(\d{2})_', p.name).group(1))
    with xr.open_dataset(p) as ds:
        eksik = [v for v in GEREKLI if v not in ds.variables]
        if eksik:
            print(f'!! GOES-{uydu:02d} atlandı, eksik değişken: {eksik}. Mevcut: {list(ds.variables)}')
            atlanan.append(p.name); continue
        d = ds[GEREKLI].to_dataframe().reset_index()
    d['status'] = d.status.astype(str).str.strip()
    d['flare_class'] = d.flare_class.astype(str).str.strip().replace({'nan': None, 'None': None, '': None})
    d['uydu'] = uydu
    parcalar.append(d)
    print(f'GOES-{uydu:02d}: {len(d):>7,} satır | {d.time.min():%Y-%m-%d} → {d.time.max():%Y-%m-%d} | '
          f'durumlar: {sorted(d.status.unique())}')
ham = pd.concat(parcalar, ignore_index=True)
print('\nToplam satır:', f'{len(ham):,}', '| atlanan dosya:', atlanan or 'yok')

## 2 · Satırları olaylara çevir
Her `uydu + flare_id` bir patlama. Pik akı ve sınıf `EVENT_PEAK` satırından, başlangıç `EVENT_START`, bitiş ve toplam enerji (`integrated_flux`) `EVENT_END` satırından alınır.

Kontroller: (1) bir olayda birden fazla pik satırı var mı, (2) bizim akıdan hesapladığımız sınıf harfi NCEI'nin yazdığı sınıfla uyuşuyor mu.

In [ ]:
olay = olaylara_cevir(ham)
olay = olay[(olay.pik >= '2000-01-01') & (olay.pik < '2026-01-01')].reset_index(drop=True)
print('Uydu bazında olay sayısı (2000–2025):')
print(olay.groupby('uydu').agg(olay=('pik', 'size'), ilk=('pik', 'min'), son=('pik', 'max')).to_string())

print('\nOlay başına pik satırı sayısı dağılımı:')
print(olay.pik_sayisi.value_counts().sort_index().to_string())

uyum = (olay.sinif_ncei.str[0] == olay.sinif)
print(f'\nSınıf harfi uyumu (bizim hesap vs NCEI): %{uyum.mean()*100:.2f}')
if (~uyum).any():
    print(olay.loc[~uyum, ['uydu', 'pik', 'pik_aki', 'sinif', 'sinif_ncei']].head(10).to_string(index=False))
print('\nEksik başlangıç/bitiş oranı (%):', (olay[['baslangic', 'bitis']].isna().mean()*100).round(2).to_dict())

## 3 · Uyduları birleştir
Aynı dönemde 2–3 uydu birlikte çalıştığı için aynı patlama birden çok kez kayıtlı. Kural: pik zamanları **10 dakika** içinde, uydular farklı, akıları en fazla 3 kat farklı olan kayıtlar tek patlama sayılır. Temsilci olarak aşağıdaki öncelik sırasında ilk gelen uydunun ölçümü alınır.

Öncelik sırası yaklaşıktır ve NOAA'nın birincil uydu tercihlerini kabaca izler. Seçimin sonucu etkileyip etkilemediğini görmek için aynı patlamayı gören uydu çiftlerinin akı oranı da raporlanır (uydular arası kalibrasyon kontrolü). Bu karar tezde metodoloji bölümünde belirtilecek.

In [ ]:
ONCELIK = {16: 1, 19: 2, 18: 3, 17: 4, 15: 5, 13: 6, 14: 7, 12: 8, 10: 9, 11: 10, 8: 11, 9: 12}
ncei, tum = uydulari_birlestir(olay, ONCELIK)
print(f'Uydu kayıtları: {len(tum):,}  →  tekil patlama: {len(ncei):,}')
print('Kaç uydunun gördüğü:'); print(ncei.n_uydu.value_counts().sort_index().to_string())

# Uydular arası kalibrasyon kontrolü: temsilciye göre diğer uyduların akı oranı (≥C1 olaylar)
t = tum.merge(ncei[['grup', 'uydu', 'pik_aki']].rename(columns={'uydu': 'temsil_uydu', 'pik_aki': 'temsil_aki'}), on='grup')
t = t[(t.uydu != t.temsil_uydu) & (t.temsil_aki >= 1e-6)]
t['oran'] = t.pik_aki / t.temsil_aki
print('\nUydu çiftleri arası akı oranı (medyan, ≥C1):')
print(t.groupby(['temsil_uydu', 'uydu']).oran.agg(['median', 'size']).round(3).to_string())

ncei['sinif_metin'] = sinif_metin(ncei.pik_aki)
ncei.loc[ncei.doymus, 'sinif_metin'] = '≥' + ncei.loc[ncei.doymus, 'sinif_metin']
ncei.to_parquet(ISLENMIS/'flare_ncei.parquet', index=False)
tum.to_parquet(ISLENMIS/'flare_ncei_tum_uydular.parquet', index=False)
print('\nKaydedildi: flare_ncei.parquet, flare_ncei_tum_uydular.parquet')
print('\nSınıf dağılımı (NCEI, tekil):'); print(ncei.sinif.value_counts().reindex(list('ABCMX')).fillna(0).astype(int).to_string())
print('\nEn güçlü 10 patlama (NCEI bilimsel kalite):')
print(ncei.nlargest(10, 'pik_aki')[['pik', 'sinif_metin', 'uydu', 'n_uydu', 'doymus']].to_string(index=False))
print(f'\nDedektörün doyduğu patlamalar ({int(ncei.doymus.sum())} adet, akı alt sınırdır, ≥ ile gösterildi):')
print(ncei[ncei.doymus].sort_values('pik')[['pik', 'sinif_metin', 'uydu', 'n_uydu']].to_string(index=False))

## 4 · Ölçek farkının ölçülmesi: NCEI / HEK akı oranı
Her NCEI patlaması (≥C1) en yakın HEK kaydıyla (±10 dk) eşleştirilir ve iki akının oranı alınır. 2024-05-08'deki hatalı X9.0 kaydı önce HEK listesinden çıkarılır.

**Beklenti:** GOES 8–15 döneminde oran ≈ 1,43 (0.7 çarpanının geri alınması), GOES-16 sonrası ≈ 1.

In [ ]:
hek = pd.read_parquet(ISLENMIS/'flare_hek.parquet')
hatali = (hek.event_peaktime == '2024-05-08 21:40:00') & (hek.fl_goescls == 'X9.0') & (hek.ar_noaanum == 0)
print('Çıkarılan hatalı HEK kaydı:', int(hatali.sum()))
hek = hek[~hatali]
hek.to_parquet(ISLENMIS/'flare_hek_temiz.parquet', index=False)

m = hek_ile_eslestir(ncei[ncei.pik_aki >= 1e-6], hek)
print(f'Eşleşme oranı: %{m.aki_hek.notna().mean()*100:.1f}')
m = m.dropna(subset=['oran'])
m['yil'] = m.pik.dt.year
yillik = m.groupby('yil').oran.agg(medyan='median', q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75), n='size')
print('\nYıllık NCEI/HEK akı oranı:'); print(yillik.round(3).to_string())
m.to_parquet(ISLENMIS/'ncei_hek_eslesme.parquet', index=False)

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.scatter(m.pik, m.oran, s=2, alpha=.15, color='#3fa0c9', label='Tek patlama')
ax.plot(pd.to_datetime(yillik.index.astype(str) + '-07-01'), yillik.medyan, 'o-', color='#c4432c', lw=2, label='Yıllık medyan')
ax.axhline(1/0.7, ls='--', color='grey'); ax.text(pd.Timestamp('2000-03-01'), 1/0.7, ' 1/0.7 = 1,43', va='bottom', color='grey')
ax.axhline(1, ls=':', color='grey')
ax.set_ylim(0.5, 2.5); ax.set_ylabel('NCEI pik akı / HEK pik akı'); ax.legend(loc='upper right'); ax.grid(alpha=.25)
ax.set_title('Operasyonel (HEK) ve bilimsel kalite (NCEI) flare akıları arasındaki ölçek farkı, ≥C1')
fig.savefig(SEKIL/'02_olcek_farki.png', dpi=200); plt.show()

## 5 · Düzeltme öncesi ve sonrası
Üst panel: leke sayısı. Orta: HEK listesiyle aylık M+X sayısı (düzeltme öncesi). Alt: NCEI listesiyle aylık M+X sayısı (düzeltme sonrası). Altta döngü bazında ≥M1 patlama sayısı ve **leke başına patlama** (Seçenek B göstergesinin ilk hali).

Not: Döngü 23 bu pencerede yalnızca 2000'den itibaren var (yükseliş evresi eksik), bu yüzden toplam sayılar doğrudan kıyaslanamaz. Leke başına oran bu etkiyi kısmen giderir.

In [ ]:
sn = pd.read_parquet(ISLENMIS/'sn_aylik_1749.parquet').set_index('tarih').sn
sn_dz = pd.read_parquet(ISLENMIS/'sn_aylik_duzlestirilmis.parquet').set_index('tarih').sn
pen = (pd.Timestamp('2000-01-01'), pd.Timestamp('2025-12-31'))
sn, sn_dz = sn[pen[0]:pen[1]], sn_dz[pen[0]:pen[1]]
hek_mx = hek[hek.sinif.isin(['M', 'X'])].set_index('event_peaktime').resample('MS').size().reindex(sn.index, fill_value=0)
nc_mx  = ncei[ncei.sinif.isin(['M', 'X'])].set_index('pik').resample('MS').size().reindex(sn.index, fill_value=0)

fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True, constrained_layout=True)
ax[0].bar(sn.index, sn, width=25, color='#f5a524', alpha=.45); ax[0].plot(sn_dz.index, sn_dz, color='#b86e00', lw=2)
ax[0].set_ylabel('Güneş lekesi sayısı')
ust = max(hek_mx.max(), nc_mx.max()) * 1.05
ax[1].bar(hek_mx.index, hek_mx, width=25, color='#8a94ad'); ax[1].set_ylabel('M+X / ay\nHEK (öncesi)'); ax[1].set_ylim(0, ust)
ax[2].bar(nc_mx.index, nc_mx, width=25, color='#3fa0c9'); ax[2].set_ylabel('M+X / ay\nNCEI (sonrası)'); ax[2].set_ylim(0, ust)
for x in ax: x.grid(alpha=.25)
ax[2].xaxis.set_major_locator(mdates.YearLocator(2)); ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.suptitle('Flare listesi homojenleştirmesi: düzeltme öncesi ve sonrası', fontsize=13)
fig.savefig(SEKIL/'02_oncesi_sonrasi.png', dpi=200); plt.show()

DONGU = {'23 (2000–2008/11)': ('2000-01-01', '2008-11-30'),
         '24 (2008/12–2019/11)': ('2008-12-01', '2019-11-30'),
         '25 (2019/12–2025)': ('2019-12-01', '2025-12-31')}
satir = []
for ad, (b, s) in DONGU.items():
    toplam_sn = sn[b:s].sum()
    h = ((hek.event_peaktime >= b) & (hek.event_peaktime <= s) & hek.sinif.isin(['M', 'X'])).sum()
    n = ((ncei.pik >= b) & (ncei.pik <= s) & ncei.sinif.isin(['M', 'X'])).sum()
    nx = ((ncei.pik >= b) & (ncei.pik <= s) & (ncei.sinif == 'X')).sum()
    satir.append([ad, h, n, nx, round(toplam_sn), round(n / toplam_sn * 1000, 2)])
dongu_tablo = pd.DataFrame(satir, columns=['Döngü', 'M+X HEK', 'M+X NCEI', 'X NCEI', 'Σ aylık SN', 'M+X / 1000 SN (NCEI)'])
dongu_tablo.to_csv(ISLENMIS/'02_dongu_ozeti.csv', index=False)
dongu_tablo